# Feature Extraction Pipeline
This pipeline holds the preprocessing of the `Standard calibration in culture media_extended.xlsx` and the feature extraction process. The results will be 3 `.csv` files, each holding the features for 3 types experiments, specific for this dataset:
1. the core features suite
2. the extended core features suite
3. the experimental features suite

### Dataset structure
- There are 42 signals, stored as column vectors
- The first signal, from the first column, represents the potential - denoted with $E$ and measured in $V$ (volts) - and this potential was applied in all experiments, ensuring the same potential window.
- The other signals represent the cureents - denoted with $I$ and measured in $\mu A$ (microampers) - characteristic to the initial potential $E$ and to the concentration values - denoted with $c$ and measured in $\mu M$ (micromolars), for which the experiment was performed.
- For each concentration were performed a number of readings, as shown bellow:

| $c$ [$\mu M$]| Number of readings |
| -------- | ------- |
| 0 | 1 (the blank signal)|
| 100 | 3 |
| 50 | 3 |
| 25 | 3 |
| 15 | 3 |
| 10 | 3 |
| 7.5 | 3 |
| 5 | 3 |
| 2.5 | 2 |
| 1 | 3 |
| 0.5 | 6 |
| 0.5 | 6 |
| 0.25 | 3 |
| 0.1 | 5 |

### The Problem we are trying to solve
- Create a ML model that predicts the concentration value (in $\mu M$) of a given voltammogram signal.
- The current signal, represented in the voltammogram, can be seen as a function of potential. Therefore, the entire problem can be written as:
$$c = I(E)$$

In [1]:
# imports

import pandas as pd
from voltammogram_signal import Signal

In [2]:
# reading the Standard calibration media dataset
df = pd.read_excel('datasets/Standard calibration in culture media_extended.xlsx', sheet_name='Raw data')
df.head()

,E0_Blank_culture media MH - moving average baseline,Unnamed: 1,Pyo 100 uM_MH - moving average baseline,Unnamed: 3,Unnamed: 4,Pyo 50 uM_MH - moving average baseline,Unnamed: 6,Unnamed: 7,Pyo 25 uM_MH - moving average baseline,Unnamed: 9,...,Unnamed: 32,Unnamed: 33,Pyo 0.25 uM_MH - moving average baseline,Unnamed: 35,Unnamed: 36,Pyo 0.1 uM_MH - moving average baseline,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41
0,V,µA,µA,µA,µA,µA,µA,µA,µA,µA,...,µA,µA,µA,µA,µA,µA,µA,µA,µA,µA
1,-0.600097,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-0.595262,0.063593,0.098656,0.109374,0.115718,0.082966,0.110628,0.104234,0.088812,0.082304,...,0.079479,0.075286,0.075833,0.076125,0.079187,0.083927,0.078385,0.110979,0.051843,0.066755
3,-0.590427,0.054031,0.121318,0.130725,0.148006,0.099352,0.122679,0.113728,0.090737,0.073225,...,0.04127,0.070401,0.071166,0.054775,0.054862,0.065697,0.060739,0.079771,0.044581,0.064057
4,-0.585592,0.033967,0.136981,0.152074,0.173293,0.115738,0.113729,0.119721,0.085662,0.085147,...,0.038062,0.048015,0.045499,0.040424,0.044536,0.043967,0.064092,0.062562,0.030319,0.047358


Constructing the Signal objects

In [ ]:
Signal.set_common_potential_E(df.iloc[1:, 0].values)
Signal.set_common_baseline_I(df.iloc[1:, 1].values)

signals = []
for sig in df.columns[1:]:
    signals.append(Signal(df[sig].values[1:]))
concentrations = [0, 100, 100, 100, 50, 50, 50, 25, 25, 25, 15, 15, 15, 10, 10, 10, 7.5, 7.5, 7.5, 5, 5, 5, 2.5, 2.5, 1, 1, 1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.25, 0.25, 0.25, 0.1, 0.1, 0.1, 0.1, 0.1]


Plotting some signals

In [8]:
signals[40].pplot(end=200)

# Feature extraction
 *   | Feature                   | Details | Reference |
| --- | ------------------------- | ----- | --------- |
| 1.  | Peak current ($I_p$)      | The peak represents the amount of electroactive species being oxidized or reduced. | Laviron, J. Electroanal. Chem. (1979) & Nicholson, Anal. Chem. (1965) |
| 2.  | Peak potential ($E_p$)    | The potential corresponding for that peak current. It represents the redox potential of pyocyanin under your experimental conditions. |
| 3.  | AUC | Peak area under the curve. This represents the total charge transferred during the redox event. $$AUC_{raw}=\int I(E) dE$$|
| 5.  | Full width at half maximum (FWHM) | Indicates how sharp or broad the peak is. Peak broadening often increases at low concentration, noisy regimes and surface effects. |
| 10.  | First derivative at peak | At an ideal maximum it would be zero. It will act as a wuality indicator. $$\frac{dI}{dk}=\frac{I[k+1]-I[k-1]}{2\Delta E}$$ | Savitzky & Golay, Analytical Chemistry (1964) |
| 11.  | Second derivative at peak                | Peak sharpness, i.e. the second derrivative at the peak. Second derivative tells you how tight the maximum is. $$\frac{d^{2}I}{dk^{2}}=\frac{I[k+1]-2I[k]+I[k-1]}{\Delta E^{2}} $$ | Savitzky & Golay, Analytical Chemistry (1964) |
